<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY1_WEEK4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qdrant-client


In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install -U FlagEmbedding

In [ ]:
pip install langchain-groq

In [ ]:
import json
import uuid
import os
from typing import List, Dict, Generator
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
from sentence_transformers import SentenceTransformer
from google.colab import userdata

In [ ]:
from FlagEmbedding import FlagAutoModel

In [ ]:
bge = FlagAutoModel.from_finetuned('BAAI/bge-m3',
                                      use_fp16=True)

In [ ]:

encoder = SentenceTransformer("intfloat/multilingual-e5-large")
VECTOR_DIMENSION = encoder.get_sentence_embedding_dimension()

qdrant_client = QdrantClient(url=userdata.get('QDRANT_URL'), api_key=userdata.get('QDRANT_API_KEY'))

In [ ]:
def search_qdrant_bge(query_text: str, search_filter: Filter = None):
    # E5 model needs "query: " prefix for questions/searches
    query_dense = bge.encode(query_text)['dense_vecs']

    # Perform the search
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_dense,
        query_filter=search_filter,
        limit=1
    )
    actual_points = results.points
    print(f"\n Results for query: '{query_text}'")
    if not results:
        print("No results found matching this filter.")
        return

    for idx, hit in enumerate(actual_points):
        print(f"\n[Result #{idx+1}]")
        print(f"-> Category: {hit.payload.get('category')}")
        print(f"-> Source File: {hit.payload.get('source_file')}")
        print(f"-> Text: {hit.payload.get('page_content')[:450]}...")

In [ ]:
collection_name = "second_policies_collection"

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

In [ ]:
def search_qdrant_multilingual(query_text: str, search_filter: Filter = None):
    # E5 model needs "query: " prefix for questions/searches
    query_vector = encoder.encode(f"query: {query_text}").tolist()

    # Perform the search
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        query_filter=search_filter,
        limit=1
    )
    actual_points = results.points
    print(f"\n Results for query: '{query_text}'")
    if not results:
        print("No results found matching this filter.")
        return

    for idx, hit in enumerate(actual_points):
        print(f"\n[Result #{idx+1}]")
        print(f"-> Category: {hit.payload.get('category')}")
        print(f"-> Source File: {hit.payload.get('source_file')}")
        print(f"-> Text: {hit.payload.get('page_content')[:450]}...")

In [ ]:
filter_2 = Filter(
    must_not=[
        FieldCondition(
            key="source_file",
            match=MatchValue(value="about.docx")
        )
    ]
)

In [ ]:
print("BGE")
search_qdrant_bge(query_text="условия возврата устройств", search_filter=filter_2)
print("\nMultilingual")
search_qdrant_multilingual(query_text="условия возврата устройств")

BGE

 Results for query: 'условия возврата устройств'

[Result #1]
-> Category: Гарантия
-> Source File: warranty.docx
-> Text: В течение срока гарантии на заводской брак, который прописан в гарантийном талоне к товару; При наличии документов, подтверждающих факт и дату покупки...

[Result #2]
-> Category: Гарантия
-> Source File: warranty.docx
-> Text: Условия возврата и обмена Если купленный товар в сети магазинов O!Store оказался с Существенным недостатком и невозможным для использования в соответс...

 Multilingual

 Results for query: 'условия возврата устройств'

[Result #1]
-> Category: Гарантия
-> Source File: warranty.docx
-> Text: В течение срока гарантии на заводской брак, который прописан в гарантийном талоне к товару; При наличии документов, подтверждающих факт и дату покупки...

[Result #2]
-> Category: Гарантия
-> Source File: warranty.docx
-> Text: Условия возврата и обмена Если купленный товар в сети магазинов O!Store оказался с Существенным недостатком и невозможным д

In [ ]:
print("BGE")
search_qdrant_bge(query_text="«В течение скольки дней клиент может вернуть аксессуар в O!Store, если он сохранил чек и товарный вид?»")
print("\nMultilingual")
search_qdrant_multilingual(query_text="«В течение скольки дней клиент может вернуть аксессуар в O!Store, если он сохранил чек и товарный вид?»")

BGE

 Results for query: '«В течение скольки дней клиент может вернуть аксессуар в O!Store, если он сохранил чек и товарный вид?»'

[Result #1]
-> Category: Гарантия
-> Source File: warranty.docx
-> Text: Условия возврата и обмена Если купленный товар в сети магазинов O!Store оказался с Существенным недостатком и невозможным для использования в соответствии с его целевым назначением, либо не может быть устранен недостаток, либо проявляется вновь после устранения, то можно вернуть уплаченные за товар деньги, либо обменять при соблюдении следующих условий: С момента покупки прошло не более 14 дней (включая день покупки); Товар не эксплуатировался, с...

Multilingual

 Results for query: '«В течение скольки дней клиент может вернуть аксессуар в O!Store, если он сохранил чек и товарный вид?»'

[Result #1]
-> Category: Гарантия
-> Source File: warranty.docx
-> Text: Android приставок O!TV) может быть осуществлён: В течение четырнадцати (14) календарных дней с момента покупки товара; При нал

In [ ]:
print("BGE")
search_qdrant_bge(query_text="В каких случаях сотрудник O!Store НЕ несёт материальную ответственность за повреждение смартфона на витрине?")
print("\nMultilingual")
search_qdrant_multilingual(query_text="В каких случаях сотрудник O!Store НЕ несёт материальную ответственность за повреждение смартфона на витрине?")

BGE

 Results for query: 'В каких случаях сотрудник O!Store НЕ несёт материальную ответственность за повреждение смартфона на витрине?'

[Result #1]
-> Category: Возврат
-> Source File: return_policy.docx
-> Text: , сохранен товарный вид и его неработоспособность подтверждается в момент обращения в O!Store для 4G-устройств компании или в Сервис-центр для телефонов; Имеются все сопутствующие продаже товара документы (паспорт, гарантийный талон, кассовый чек); *К существенному недостатку товара не относятся и товар не подлежит обмену/возврату на аналогичный, если товар не подошел по форме, габаритам, расцветке или по иным причинам. Обмен и возврат смартфонов...

Multilingual

 Results for query: 'В каких случаях сотрудник O!Store НЕ несёт материальную ответственность за повреждение смартфона на витрине?'

[Result #1]
-> Category: Возврат
-> Source File: return_policy.docx
-> Text: , сохранен товарный вид и его неработоспособность подтверждается в момент обращения в O!Store для 4G-устройс

In [ ]:
print("BGE")
search_qdrant_bge(query_text="Каков график инкассации наличных для региональных точек, например, O!Store Кара-Балта?")
print("\nMultilingual")
search_qdrant_multilingual(query_text="Каков график инкассации наличных для региональных точек, например, O!Store Кара-Балта?")

BGE

 Results for query: 'Каков график инкассации наличных для региональных точек, например, O!Store Кара-Балта?'

[Result #1]
-> Category: FAQ
-> Source File: FAQ.docx
-> Text: ку. Инструкция по оплате через  О!Деньги (если вы открыли десктопную версию сайта): При выборе способа оплаты через О!Деньги и подтверждения заказа на вашем экране появится QR-код. Откройте приложение “Мой О!”, раздел О!Деньги и нажмите на сканер штрихкодов и QR. Отсканируйте QR-код, выберите нужный источник средств и нажмите на кнопку “Подтвердить оплату”. Инструкция по оплате через  О!Деньги (если вы открыли мобильную версию сайта): При выборе ...

Multilingual

 Results for query: 'Каков график инкассации наличных для региональных точек, например, O!Store Кара-Балта?'

[Result #1]
-> Category: Магазины
-> Source File: stores_info.docx
-> Text: н-Вс(13:00-14:00) город Кочкор АтаКочкор Ата: ул. Гагарина, б\н (ориентир сберкасса "Айыл Банк"), График работы: Пн-Вс(09:00-19:00)Перерыв Пн-Вс(13:00-14:00) город Май

In [41]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature=0,
    groq_api_key=userdata.get('GROQ_API_KEY'),
    model_name="llama-3.3-70b-versatile"
)

In [44]:
from typing import Literal
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END

def retrieve_node(state: MessagesState):
    """
    Нода поиска. Исправлен вызов bge.encode для FlagEmbedding.
    """
    user_question = state["messages"][-1].content
    retrieved_chunks = []

    try:
        query_dense_bge = bge.encode(user_question)['dense_vecs']
        results_bge = qdrant_client.query_points(
            collection_name=collection_name,
            query=query_dense_bge,
            limit=1
        )
        if results_bge.points:
            chunk_text = results_bge.points[0].payload.get('page_content', '')
            retrieved_chunks.append(f"[Контекст из BGE-M3]: {chunk_text}")
    except Exception as e:
        print(f"Ошибка поиска BGE-M3: {e}")

    try:
        query_vector_e5 = encoder.encode(f"query: {user_question}").tolist()
        results_e5 = qdrant_client.query_points(
            collection_name=collection_name,
            query=query_vector_e5,
            limit=1
        )
        if results_e5.points:
            chunk_text = results_e5.points[0].payload.get('page_content', '')
            retrieved_chunks.append(f"[Контекст из Multilingual-E5]: {chunk_text}")
    except Exception as e:
        print(f"Ошибка поиска E5: {e}")

    context_str = "\n\n".join(retrieved_chunks) if retrieved_chunks else "Контекст в базе данных отсутствует."

    return {
        "messages": [AIMessage(content=context_str, name="context_holder")]
    }


def generate_node(state: MessagesState):
    """
    Нода генерации. Формирует финальный ответ через Groq LLM.
    """
    context = state["messages"][-1].content
    user_question = state["messages"][-2].content

    system_prompt = (
        "Вы — корпоративный AI-ассистент поддержки сотрудников O!Store.\n"
        "Ваша задача — строго отвечать на вопросы пользователя на основе предоставленного контекста.\n"
        "Если в контексте нет четкого ответа на вопрос или модели поиска вернули ошибочные данные, "
        "вы ОБЯЗАНЫ строго ответить: 'К сожалению, я не могу ответить на этот вопрос на основе имеющихся регламентов O!Store.'\n"
        "Не придумывайте факты, не используйте внешние знания, которых нет в тексте.\n\n"
        f"ПРЕДОСТАВЛЕННЫЙ КОНТЕКСТ:\n{context}"
    )

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_question)
    ])

    return {
        "messages": [AIMessage(content=response.content)]
    }


workflow = StateGraph(MessagesState)

workflow.add_node("retriever", retrieve_node)
workflow.add_node("generator", generate_node)

workflow.add_edge(START, "retriever")
workflow.add_edge("retriever", "generator")
workflow.add_edge("generator", END)

app = workflow.compile()

In [45]:
inputs = {"messages": [HumanMessage(content="Каковы правила инкассации для удаленных точек?")]}
config = {"configurable": {"thread_id": "groq_session_1"}}

print("--- Запуск графа LangGraph + Groq ---")
for output in app.stream(inputs, config, stream_mode="values"):
    last_message = output["messages"][-1]
    if last_message.name != "context_holder":
        print(f"[{last_message.type.upper()}]: {last_message.content}\n")

--- Запуск графа LangGraph + Groq ---
[HUMAN]: Каковы правила инкассации для удаленных точек?

[AI]: К сожалению, я не могу ответить на этот вопрос на основе имеющихся регламентов O!Store.



In [46]:
inputs = {"messages": [HumanMessage(content="В течение скольки дней клиент может вернуть товар?")]}
config = {"configurable": {"thread_id": "groq_session_test_2"}}

print("--- Тест №1: Сроки возврата ---")
for output in app.stream(inputs, config, stream_mode="values"):
    last_message = output["messages"][-1]
    if last_message.name != "context_holder":
        print(f"[{last_message.type.upper()}]: {last_message.content}\n")

--- Тест №1: Сроки возврата ---
[HUMAN]: В течение скольки дней клиент может вернуть товар?

[AI]: Клиент может вернуть товар в течение 14 календарных дней с момента покупки.



In [47]:
inputs = {"messages": [HumanMessage(content="Каков график работы или инкассации в регионах, например Кара-Балта?")]}
config = {"configurable": {"thread_id": "groq_session_test_3"}}

print("--- Тест №2: Специфика регионов ---")
for output in app.stream(inputs, config, stream_mode="values"):
    last_message = output["messages"][-1]
    if last_message.name != "context_holder":
        print(f"[{last_message.type.upper()}]: {last_message.content}\n")

--- Тест №2: Специфика регионов ---
[HUMAN]: Каков график работы или инкассации в регионах, например Кара-Балта?

[AI]: К сожалению, я не могу ответить на этот вопрос на основе имеющихся регламентов O!Store.

